# Step 1: Create Mapping Table

Replicates `code/create_mapping_table.do`

**Input:** `raw/mapping_table_final.xlsx` (sheet: mapping)  
**Output:** `output_python/mapping_table_fy_final.{dta,parquet,csv}`  
**Parity baseline:** `output/mapping_table_fy_final.dta`

In [1]:
import pandas as pd
import numpy as np
import pyreadstat
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Paths
ROOT = Path.cwd().parent  # assumes notebook runs from notebooks/
RAW = ROOT / 'raw'
OUTPUT_STATA = ROOT / 'output'
OUTPUT_PY = ROOT / 'output_python'
OUTPUT_PY.mkdir(exist_ok=True)

print(f'Root: {ROOT}')
print(f'Raw:  {RAW}')
print(f'Stata baseline: {OUTPUT_STATA}')
print(f'Python output:  {OUTPUT_PY}')

Root: /Users/Work/Desktop/Pension Research/ppd-data
Raw:  /Users/Work/Desktop/Pension Research/ppd-data/raw
Stata baseline: /Users/Work/Desktop/Pension Research/ppd-data/output
Python output:  /Users/Work/Desktop/Pension Research/ppd-data/output_python


## 1. Import mapping sheet and drop `notes`

Stata: `import excel ... sheet("mapping") firstrow` then `drop notes`

In [2]:
df = pd.read_excel(RAW / 'mapping_table_final.xlsx', sheet_name='mapping')
if 'notes' in df.columns:
    df = df.drop(columns=['notes'])
print(f'Rows after import: {len(df)}')
print(f'Columns: {list(df.columns)}')
df.head()

Rows after import: 237
Columns: ['ppd_id', 'ppd_name', 'startfy', 'endfy', 'pub_id', 'pub_system_name', 'preqin_id', 'preqin_name', 'tr13fid1', 'tr13fname1', 'tr13fid2', 'tr13fname2', 'tr13fid3', 'tr13fname3', 'tr13fid4', 'tr13fname4']


,ppd_id,ppd_name,startfy,endfy,pub_id,pub_system_name,preqin_id,preqin_name,tr13fid1,tr13fname1,tr13fid2,tr13fname2,tr13fid3,tr13fname3,tr13fid4,tr13fname4
0,1,Alabama ERS,2001,2023,AL1001,Retirement Systems of Alabama,2711.0,Retirement Systems of Alabama,9059.0,Retirement Systems Of Alabama,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Alabama Teachers,2001,2023,AL1001,Retirement Systems of Alabama,2711.0,Retirement Systems of Alabama,9059.0,Retirement Systems Of Alabama,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Alaska PERS,2001,2023,AK1001,Alaska Retirement Management Board,2326.0,Alaska Retirement Management Board,18936.0,Alaska Retirement Mgmt Bd,NaN,NaN,NaN,NaN,NaN,NaN
3,4,Alaska Teachers,2001,2023,AK1001,Alaska Retirement Management Board,2326.0,Alaska Retirement Management Board,18936.0,Alaska Retirement Mgmt Bd,NaN,NaN,NaN,NaN,NaN,NaN
4,5,Arizona Public Safety,2001,2023,AZ1001,Arizona Public Safety Personnel Retirement System,3714.0,Arizona Public Safety Personnel Retirement System,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Generate pair_id and expand year range

Stata logic:
```stata
egen pair_id = group(pub_id ppd_id)
expand 2, generate(index)
sum startfy → gen fy = r(min) if index == 0
sum endfy  → replace fy = r(max) if index == 1
drop index
xtset pair_id fy
tsfill, full
```
Creates two rows per pair (one at global min startfy, one at global max endfy), then fills every year in between for each pair.

In [11]:
# Create pair_id (unique per pub_id x ppd_id combination)
df['pair_id'] = df.groupby(['pub_id', 'ppd_id']).ngroup() + 1

# Global min/max fiscal years (Stata: sum startfy → r(min); sum endfy → r(max))
fy_min = int(df['startfy'].min())
fy_max = int(df['endfy'].max())
print(f'Global FY range: {fy_min} – {fy_max}')

# --- Mirror Stata exactly ---
# expand 2, generate(index): duplicate each row; index=0 for originals, index=1 for copies
df0 = df.copy(); df0['index'] = 0
df1 = df.copy(); df1['index'] = 1
df_expanded = pd.concat([df0, df1], ignore_index=True)

# gen fy = r(min) if index == 0; replace fy = r(max) if index == 1
df_expanded['fy'] = np.where(df_expanded['index'] == 0, fy_min, fy_max)
df_expanded = df_expanded.drop(columns=['index'])

# xtset pair_id fy / tsfill, full: for each pair, fill all years in [fy_min, fy_max]
# Create complete panel skeleton and merge boundary data to preserve NaN structure
all_pairs = df_expanded[['pair_id']].drop_duplicates()
all_fy = pd.DataFrame({'fy': range(fy_min, fy_max + 1)})
skeleton = all_pairs.merge(all_fy, how='cross')

# Merge boundary-year data onto the skeleton (only boundary years get values)
df_filled = skeleton.merge(df_expanded, on=['pair_id', 'fy'], how='left')
df_filled = df_filled.sort_values(['pair_id', 'fy']).reset_index(drop=True)
print(f'Rows after tsfill: {len(df_filled)}')

Global FY range: 2001 – 2023
Rows after tsfill: 5451


## 3. Forward-fill missing values within each pair

Stata: `foreach var of varlist ppd_id-preqin_name { bysort pair_id: replace var = var[_n-1] if missing(var) }`

In [12]:
# Stata: foreach var of varlist ppd_id-preqin_name {
#     bysort pair_id: replace `var' = `var'[_n-1] if missing(`var')
# }
# The varlist range ppd_id-preqin_name includes ONLY:
#   ppd_id, ppd_name, startfy, endfy, pub_id, pub_system_name, preqin_id, preqin_name
# The tr13f* columns are NOT forward-filled.

ffill_cols = ['ppd_id', 'ppd_name', 'startfy', 'endfy',
              'pub_id', 'pub_system_name', 'preqin_id', 'preqin_name']
df_filled[ffill_cols] = df_filled.groupby('pair_id')[ffill_cols].ffill()
print(f'Nulls in ffill cols:     {df_filled[ffill_cols].isnull().sum().sum()}')
tr13f_cols = [c for c in df_filled.columns if c.startswith('tr13f')]
print(f'Nulls in tr13f cols:     {df_filled[tr13f_cols].isnull().sum().sum()}')

Nulls in ffill cols:     1426
Nulls in tr13f cols:     43376


## 4. Keep only rows where fy is within [startfy, endfy], then drop startfy/endfy

Stata: `keep if inrange(fy, startfy, endfy)` then `drop startfy endfy`

In [13]:
df_filtered = df_filled[
    (df_filled['fy'] >= df_filled['startfy']) &
    (df_filled['fy'] <= df_filled['endfy'])
].copy()
df_filtered = df_filtered.drop(columns=['startfy', 'endfy'])
print(f'Rows after inrange filter: {len(df_filtered)}')

Rows after inrange filter: 5244


## 5. Final cleanup: drop pair_id, reorder columns, sort

Stata:
```stata
drop pair_id
order fy ppd_id ppd_name pub_id pub_system_name preqin_id preqin_name
sort ppd_id fy
```

In [14]:
df_final = df_filtered.drop(columns=['pair_id'])

# Enforce Stata column order: leading cols first, then any remaining
lead_cols = ['fy', 'ppd_id', 'ppd_name', 'pub_id', 'pub_system_name',
             'preqin_id', 'preqin_name']
other_cols = [c for c in df_final.columns if c not in lead_cols]
df_final = df_final[lead_cols + other_cols]

# Sort by ppd_id then fy (Stata: sort ppd_id fy)
df_final = df_final.sort_values(['ppd_id', 'fy']).reset_index(drop=True)

print(f'Final shape: {df_final.shape}')
print(f'Columns: {list(df_final.columns)}')
df_final.head(10)

Final shape: (5244, 15)
Columns: ['fy', 'ppd_id', 'ppd_name', 'pub_id', 'pub_system_name', 'preqin_id', 'preqin_name', 'tr13fid1', 'tr13fname1', 'tr13fid2', 'tr13fname2', 'tr13fid3', 'tr13fname3', 'tr13fid4', 'tr13fname4']


,fy,ppd_id,ppd_name,pub_id,pub_system_name,preqin_id,preqin_name,tr13fid1,tr13fname1,tr13fid2,tr13fname2,tr13fid3,tr13fname3,tr13fid4,tr13fname4
0,2001,1.0,Alabama ERS,AL1001,Retirement Systems of Alabama,2711.0,Retirement Systems of Alabama,9059.0,Retirement Systems Of Alabama,NaN,NaN,NaN,NaN,NaN,NaN
1,2002,1.0,Alabama ERS,AL1001,Retirement Systems of Alabama,2711.0,Retirement Systems of Alabama,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2003,1.0,Alabama ERS,AL1001,Retirement Systems of Alabama,2711.0,Retirement Systems of Alabama,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2004,1.0,Alabama ERS,AL1001,Retirement Systems of Alabama,2711.0,Retirement Systems of Alabama,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2005,1.0,Alabama ERS,AL1001,Retirement Systems of Alabama,2711.0,Retirement Systems of Alabama,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2006,1.0,Alabama ERS,AL1001,Retirement Systems of Alabama,2711.0,Retirement Systems of Alabama,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2007,1.0,Alabama ERS,AL1001,Retirement Systems of Alabama,2711.0,Retirement Systems of Alabama,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2008,1.0,Alabama ERS,AL1001,Retirement Systems of Alabama,2711.0,Retirement Systems of Alabama,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2009,1.0,Alabama ERS,AL1001,Retirement Systems of Alabama,2711.0,Retirement Systems of Alabama,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2010,1.0,Alabama ERS,AL1001,Retirement Systems of Alabama,2711.0,Retirement Systems of Alabama,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 6. Save outputs (DTA, Parquet, CSV)

In [15]:
# Save DTA
pyreadstat.write_dta(
    df_final,
    OUTPUT_PY / 'mapping_table_fy_final.dta'
)

# Save Parquet
df_final.to_parquet(OUTPUT_PY / 'mapping_table_fy_final.parquet', index=False)

# Save CSV
df_final.to_csv(OUTPUT_PY / 'mapping_table_fy_final.csv', index=False)

print('Saved: mapping_table_fy_final.{dta, parquet, csv}')

Saved: mapping_table_fy_final.{dta, parquet, csv}


## 7. Parity Check vs Stata Baseline

Load `output/mapping_table_fy_final.dta` and compare row counts, key uniqueness, column set, null profiles, and values.

In [17]:
# Load Stata baseline (use pandas reader for broader format support)
stata_df = pd.read_stata(OUTPUT_STATA / 'mapping_table_fy_final.dta')
stata_df = stata_df.sort_values(['ppd_id', 'fy']).reset_index(drop=True)

py_df = df_final.copy()

# --- 1. Row count ---
print(f'Row count — Stata: {len(stata_df)}, Python: {len(py_df)}, Match: {len(stata_df) == len(py_df)}')

# --- 2. Column set ---
stata_cols = set(stata_df.columns)
py_cols = set(py_df.columns)
print(f'Columns in Stata only:  {stata_cols - py_cols}')
print(f'Columns in Python only: {py_cols - stata_cols}')
common_cols = sorted(stata_cols & py_cols)
print(f'Common columns ({len(common_cols)}): {common_cols}')

# --- 3. Key uniqueness ---
py_dups = py_df.duplicated(subset=['ppd_id', 'fy']).sum()
stata_dups = stata_df.duplicated(subset=['ppd_id', 'fy']).sum()
print(f'Duplicate keys — Stata: {stata_dups}, Python: {py_dups}')

# --- 4. Null profiles (normalize: Stata stores missing strings as "", Python as NaN) ---
print('\nNull counts per column (after normalizing Stata "" → NaN for strings):')

# Normalize Stata empty strings to NaN for fair comparison
stata_norm = stata_df[common_cols].copy()
for col in stata_norm.select_dtypes(include='object').columns:
    stata_norm[col] = stata_norm[col].replace('', np.nan)

py_norm = py_df[common_cols].copy()

null_compare = pd.DataFrame({
    'stata_nulls': stata_norm.isnull().sum(),
    'python_nulls': py_norm.isnull().sum()
})
null_compare['match'] = null_compare['stata_nulls'] == null_compare['python_nulls']
print(null_compare.to_string())

# --- 5. Value comparison (common columns, aligned by ppd_id+fy) ---
merged = py_norm.merge(
    stata_norm,
    on=['ppd_id', 'fy'],
    suffixes=('_py', '_stata'),
    how='outer',
    indicator=True
)
print(f'\nMerge indicator:\n{merged["_merge"].value_counts().to_string()}')

# Numeric and string comparison for common cols
mismatches = {}
for col in common_cols:
    if col in ('ppd_id', 'fy'):
        continue
    py_col = f'{col}_py' if f'{col}_py' in merged.columns else col
    st_col = f'{col}_stata' if f'{col}_stata' in merged.columns else col
    if py_col in merged.columns and st_col in merged.columns:
        if pd.api.types.is_numeric_dtype(merged[py_col]):
            both_valid = merged[py_col].notna() & merged[st_col].notna()
            diff = (merged.loc[both_valid, py_col] - merged.loc[both_valid, st_col]).abs()
            n_diff = (diff > 1e-6).sum()
            # Also check NaN alignment
            nan_mismatch = (merged[py_col].isna() != merged[st_col].isna()).sum()
            if n_diff > 0 or nan_mismatch > 0:
                mismatches[col] = {'value_diff': int(n_diff), 'nan_mismatch': int(nan_mismatch)}
        else:
            both_valid = merged[py_col].notna() & merged[st_col].notna()
            n_diff = (merged.loc[both_valid, py_col].astype(str) != merged.loc[both_valid, st_col].astype(str)).sum()
            nan_mismatch = (merged[py_col].isna() != merged[st_col].isna()).sum()
            if n_diff > 0 or nan_mismatch > 0:
                mismatches[col] = {'value_diff': int(n_diff), 'nan_mismatch': int(nan_mismatch)}

if mismatches:
    print(f'\n⚠ Value mismatches: {mismatches}')
else:
    print('\n✅ All values match across common columns!')

print('\n--- PARITY SUMMARY ---')
all_pass = (
    len(stata_df) == len(py_df)
    and py_dups == 0
    and null_compare['match'].all()
    and len(mismatches) == 0
)
print(f'PARITY: {"✅ PASS" if all_pass else "❌ FAIL"}')

Row count — Stata: 5244, Python: 5244, Match: True
Columns in Stata only:  set()
Columns in Python only: set()
Common columns (15): ['fy', 'ppd_id', 'ppd_name', 'preqin_id', 'preqin_name', 'pub_id', 'pub_system_name', 'tr13fid1', 'tr13fid2', 'tr13fid3', 'tr13fid4', 'tr13fname1', 'tr13fname2', 'tr13fname3', 'tr13fname4']
Duplicate keys — Stata: 0, Python: 0

Null counts per column (after normalizing Stata "" → NaN for strings):
                 stata_nulls  python_nulls  match
fy                         0             0   True
ppd_id                     0             0   True
ppd_name                   0             0   True
preqin_id                637           637   True
preqin_name              637           637   True
pub_id                     0             0   True
pub_system_name            0             0   True
tr13fid1                5147          5147   True
tr13fid2                5238          5238   True
tr13fid3                5238          5238   True
tr13fid4           

In [19]:
## Deep parity: row-level spot checks and column order verification

# Reload fresh
stata_df = pd.read_stata(OUTPUT_STATA / 'mapping_table_fy_final.dta')
stata_df = stata_df.sort_values(['ppd_id', 'fy']).reset_index(drop=True)
py_df = df_final.sort_values(['ppd_id', 'fy']).reset_index(drop=True)

# Normalize Stata empty strings → NaN for string columns
for col in stata_df.select_dtypes(include='object').columns:
    stata_df[col] = stata_df[col].replace('', np.nan)

results = []

# 1. Column ORDER check
col_order_match = list(stata_df.columns) == list(py_df.columns)
results.append(('Column order match', col_order_match))

# 2. Spot checks: first, last, middle, specific rows
for label, idx in [('First row', 0), ('Last row', len(stata_df)-1), ('Row 1000', 1000), ('Row 2500', 2500)]:
    s_row = stata_df.iloc[idx]
    p_row = py_df.iloc[idx]
    all_ok = True
    bad_cols = []
    for c in stata_df.columns:
        sv, pv = s_row[c], p_row[c]
        if pd.isna(sv) and pd.isna(pv):
            continue
        elif pd.api.types.is_numeric_dtype(stata_df[c]):
            if pd.notna(sv) and pd.notna(pv) and abs(float(sv) - float(pv)) < 1e-6:
                continue
            else:
                all_ok = False; bad_cols.append((c, sv, pv))
        else:
            if str(sv) == str(pv):
                continue
            else:
                all_ok = False; bad_cols.append((c, sv, pv))
    results.append((f'{label} (ppd_id={s_row["ppd_id"]}, fy={s_row["fy"]})', all_ok))
    if not all_ok:
        for c, sv, pv in bad_cols:
            print(f'  MISMATCH in {label}: {c} → Stata={sv!r}, Python={pv!r}')

# 3. tr13f NaN structure: values should only exist at boundary years
tr13f_ok = True
tr13f_cols = [c for c in py_df.columns if c.startswith('tr13f')]
for c in tr13f_cols:
    s_years = set(stata_df.loc[stata_df[c].notna(), 'fy'].unique())
    p_years = set(py_df.loc[py_df[c].notna(), 'fy'].unique())
    if s_years != p_years:
        tr13f_ok = False
        print(f'  tr13f year mismatch for {c}: Stata={sorted(s_years)}, Py={sorted(p_years)}')
results.append(('tr13f NaN structure', tr13f_ok))

# 4. Exhaustive cell-by-cell comparison
total_mismatches = 0
bad_detail = []
for c in stata_df.columns:
    s = stata_df[c]; p = py_df[c]
    if pd.api.types.is_numeric_dtype(s):
        both_valid = s.notna() & p.notna()
        val_bad = ((s[both_valid] - p[both_valid]).abs() > 1e-6).sum()
        nan_bad = (s.isna() != p.isna()).sum()
    else:
        both_valid = s.notna() & p.notna()
        val_bad = (s[both_valid].astype(str) != p[both_valid].astype(str)).sum()
        nan_bad = (s.isna() != p.isna()).sum()
    n_bad = val_bad + nan_bad
    total_mismatches += n_bad
    if n_bad > 0:
        bad_detail.append(f'{c}: {n_bad}')
results.append((f'Exhaustive cell comparison (0 mismatches)', total_mismatches == 0))
if bad_detail:
    for d in bad_detail:
        print(f'  Cell mismatch: {d}')

# Print summary
print('\n' + '='*50)
print('DEEP PARITY DOUBLE-CHECK')
print('='*50)
for test, passed in results:
    print(f'  {"✅" if passed else "❌"} {test}')
print('='*50)
overall = all(p for _, p in results)
print(f'  OVERALL: {"✅ EXACT PARITY CONFIRMED" if overall else "❌ PARITY ISSUES FOUND"}')
print('='*50)


DEEP PARITY DOUBLE-CHECK
  ✅ Column order match
  ✅ First row (ppd_id=1, fy=2001.0)
  ✅ Last row (ppd_id=234, fy=2023.0)
  ✅ Row 1000 (ppd_id=44, fy=2012.0)
  ✅ Row 2500 (ppd_id=109, fy=2017.0)
  ✅ tr13f NaN structure
  ✅ Exhaustive cell comparison (0 mismatches)
  OVERALL: ✅ EXACT PARITY CONFIRMED
